In [1]:
import pandas as pd
import numpy as np

In [2]:
equity_df = pd.read_csv('equity_value_data_v2.csv')
equity_df

,timestamp,close_equity,user_id,date,daily_equity_change
0,2016-08-18T00:00:00Z,1211.6055,0012db34aa7b083f5714e7831195e54d,2016-08-18,NaN
1,2016-08-19T00:00:00Z,1173.5640,0012db34aa7b083f5714e7831195e54d,2016-08-19,-38.0415
2,2016-08-22T00:00:00Z,1253.0597,0012db34aa7b083f5714e7831195e54d,2016-08-22,79.4957
3,2016-08-23T00:00:00Z,1252.9050,0012db34aa7b083f5714e7831195e54d,2016-08-23,-0.1547
4,2016-08-24T00:00:00Z,1262.1360,0012db34aa7b083f5714e7831195e54d,2016-08-24,9.2310
...,...,...,...,...,...
1119153,2017-08-14T00:00:00Z,2156.2400,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-14,47.9100
1119154,2017-08-15T00:00:00Z,2134.7100,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-15,-21.5300
1119155,2017-08-16T00:00:00Z,2152.1200,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-16,17.4100
1119156,2017-08-17T00:00:00Z,2042.2800,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-17,-109.8400


In [3]:
# Extend each user's data to the end of the dataset - don't let them off the hook if they stopped trading altogether

# Make sure 'date' is a datetime
equity_df['date'] = pd.to_datetime(equity_df['date'])

# Determine the global last date
max_date = equity_df['date'].max()

# Function to expand a single user's data
def expand_user(user_df):
    user_df = user_df.sort_values('date')
    first_day = user_df['date'].min()
    all_days = pd.date_range(start=first_day, end=max_date, freq='D')
    
    expanded = pd.DataFrame({'date': all_days})
    expanded['user_id'] = user_df['user_id'].iloc[0]
    
    # Merge existing data (timestamp, close_equity, daily_equity_change)
    expanded = expanded.merge(user_df, on=['user_id','date'], how='left')
    
    # Fill missing values for inactive days
    expanded['close_equity'] = expanded['close_equity'].fillna(9)  # your inactivity assumption
    expanded['daily_equity_change'] = expanded['daily_equity_change'].fillna(0)  # no change on inactive days
    
    return expanded

# Apply to all users
full_equity_df = equity_df.groupby('user_id', group_keys=False).apply(expand_user)

# Binary flag for active days
full_equity_df['active_day'] = full_equity_df['timestamp'].notna().astype(int)

# Drop timestamp as it no longer has useful information after date was split off.
full_equity_df.drop('timestamp', axis=1, inplace=True)

full_equity_df

/var/folders/cf/4zrkk0lx1nbdx9tpklpk4jc00000gn/T/ipykernel_3679/289616640.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  full_equity_df = equity_df.groupby('user_id', group_keys=False).apply(expand_user)


,date,user_id,close_equity,daily_equity_change,active_day
0,2016-08-18,0012db34aa7b083f5714e7831195e54d,1211.6055,0.0000,1
1,2016-08-19,0012db34aa7b083f5714e7831195e54d,1173.5640,-38.0415,1
2,2016-08-20,0012db34aa7b083f5714e7831195e54d,9.0000,0.0000,0
3,2016-08-21,0012db34aa7b083f5714e7831195e54d,9.0000,0.0000,0
4,2016-08-22,0012db34aa7b083f5714e7831195e54d,1253.0597,79.4957,1
...,...,...,...,...,...
294,2017-08-14,ffc1e622f3a0b2666f09a6dcb7f27918,2156.2400,47.9100,1
295,2017-08-15,ffc1e622f3a0b2666f09a6dcb7f27918,2134.7100,-21.5300,1
296,2017-08-16,ffc1e622f3a0b2666f09a6dcb7f27918,2152.1200,17.4100,1
297,2017-08-17,ffc1e622f3a0b2666f09a6dcb7f27918,2042.2800,-109.8400,1


In [5]:
full_equity_df = full_equity_df.sort_values(['user_id', 'date'])

# Create a group that increments every time we hit an active day
grp = (
    full_equity_df['active_day']
    .eq(1)
    .groupby(full_equity_df['user_id'])
    .cumsum()
)

# Count days since last active day
full_equity_df['days_since_last_active'] = (
    full_equity_df
    .groupby(['user_id', grp])
    .cumcount()
)

In [6]:
full_equity_df['streak_at_least_28'] = (full_equity_df['days_since_last_active'] >= 28).astype(int)

In [7]:
# Sort by user and date first
full_equity_df = full_equity_df.sort_values(['user_id', 'date'])

# Use transform instead of apply
full_equity_df['ever_above_10_prev'] = full_equity_df.groupby('user_id')['close_equity'] \
                                                .transform(lambda x: x.shift(1).ge(10).cummax())

# Fill NaN for the first row per user
full_equity_df['ever_above_10_prev'] = full_equity_df['ever_above_10_prev'].fillna(False)

full_equity_df['ever_above_10_prev'] = full_equity_df['ever_above_10_prev'].astype(int)


# Daily change column
# Make sure date is datetime and data is sorted
full_equity_df['date'] = pd.to_datetime(full_equity_df['date'])

full_equity_df = full_equity_df.sort_values(['user_id', 'date'])

# Daily change in close_equity
full_equity_df['daily_equity_change'] = full_equity_df.groupby('user_id')['close_equity'].diff()


full_equity_df

,date,user_id,close_equity,daily_equity_change,active_day,days_since_last_active,streak_at_least_28,ever_above_10_prev
0,2016-08-18,0012db34aa7b083f5714e7831195e54d,1211.6055,NaN,1,0,0,0
1,2016-08-19,0012db34aa7b083f5714e7831195e54d,1173.5640,-38.0415,1,0,0,1
2,2016-08-20,0012db34aa7b083f5714e7831195e54d,9.0000,-1164.5640,0,1,0,1
3,2016-08-21,0012db34aa7b083f5714e7831195e54d,9.0000,0.0000,0,2,0,1
4,2016-08-22,0012db34aa7b083f5714e7831195e54d,1253.0597,1244.0597,1,0,0,1
...,...,...,...,...,...,...,...,...
294,2017-08-14,ffc1e622f3a0b2666f09a6dcb7f27918,2156.2400,2147.2400,1,0,0,1
295,2017-08-15,ffc1e622f3a0b2666f09a6dcb7f27918,2134.7100,-21.5300,1,0,0,1
296,2017-08-16,ffc1e622f3a0b2666f09a6dcb7f27918,2152.1200,17.4100,1,0,0,1
297,2017-08-17,ffc1e622f3a0b2666f09a6dcb7f27918,2042.2800,-109.8400,1,0,0,1


In [8]:
# Change since last active day
import numpy as np

# Ensure proper ordering
full_equity_df = full_equity_df.sort_values(['user_id', 'date'])

# Equity only on active days (NaN otherwise)
active_equity = full_equity_df['close_equity'].where(full_equity_df['active_day'] == 1)

# Previous active-day equity per user
prev_active_equity = (
    active_equity
    .groupby(full_equity_df['user_id'])
    .shift(1)
)

# Change since last active day (only for active days)
full_equity_df['change_since_last_active_day'] = np.where(
    full_equity_df['active_day'] == 1,
    full_equity_df['close_equity'] - prev_active_equity,
    np.nan
)

In [9]:
full_equity_df

,date,user_id,close_equity,daily_equity_change,active_day,days_since_last_active,streak_at_least_28,ever_above_10_prev,change_since_last_active_day
0,2016-08-18,0012db34aa7b083f5714e7831195e54d,1211.6055,NaN,1,0,0,0,NaN
1,2016-08-19,0012db34aa7b083f5714e7831195e54d,1173.5640,-38.0415,1,0,0,1,-38.0415
2,2016-08-20,0012db34aa7b083f5714e7831195e54d,9.0000,-1164.5640,0,1,0,1,NaN
3,2016-08-21,0012db34aa7b083f5714e7831195e54d,9.0000,0.0000,0,2,0,1,NaN
4,2016-08-22,0012db34aa7b083f5714e7831195e54d,1253.0597,1244.0597,1,0,0,1,NaN
...,...,...,...,...,...,...,...,...,...
294,2017-08-14,ffc1e622f3a0b2666f09a6dcb7f27918,2156.2400,2147.2400,1,0,0,1,NaN
295,2017-08-15,ffc1e622f3a0b2666f09a6dcb7f27918,2134.7100,-21.5300,1,0,0,1,-21.5300
296,2017-08-16,ffc1e622f3a0b2666f09a6dcb7f27918,2152.1200,17.4100,1,0,0,1,17.4100
297,2017-08-17,ffc1e622f3a0b2666f09a6dcb7f27918,2042.2800,-109.8400,1,0,0,1,-109.8400


In [11]:
full_equity_df['user_id'].nunique()

5584

# Churned users DataFrame

Previous incorrect version, which ended each user's data on their final active day, found only 275 churned users. The corrected version has 1005. Out of 5584 total users, this creates a churn rate of 17.9%.

In [10]:
churned_users_df = full_equity_df[full_equity_df['streak_at_least_28'] == 1].copy()
churned_users_df = churned_users_df[churned_users_df['ever_above_10_prev'] == 1]
churned_users_df.drop_duplicates(subset=['user_id'], keep='first', inplace=True)
churned_users_df

,date,user_id,close_equity,daily_equity_change,active_day,days_since_last_active,streak_at_least_28,ever_above_10_prev,change_since_last_active_day
87,2017-03-27,00440034cc4152bfb01b30f5c381c4e3,9.0,0.0,0,28,1,1,NaN
133,2017-05-26,004aab1640f3a04b87b1f404fb4c018d,9.0,0.0,0,28,1,1,NaN
160,2017-08-07,004ea9d7662aa8dc840bbff212cfa4b8,9.0,0.0,0,28,1,1,NaN
106,2016-12-02,005d630a68b4ab3a2f4cd49d9a87c50d,9.0,0.0,0,28,1,1,NaN
303,2017-06-16,00f89f56f25989b0bb7ea05bac2dccc4,9.0,0.0,0,28,1,1,NaN
...,...,...,...,...,...,...,...,...,...
199,2017-05-01,ff0ae95285c43e3a5af84860bffaa544,9.0,0.0,0,28,1,1,NaN
34,2016-10-27,ff377467d4e28b425266a8b2c8b2f5c7,9.0,0.0,0,28,1,1,NaN
228,2017-04-03,ff6d64d75fa2ffd703dabf66b7b86b99,9.0,0.0,0,28,1,1,NaN
93,2017-02-09,ff7610fdd7ac5cbfa0b17aca53af5db4,9.0,0.0,0,28,1,1,NaN


# Features


In [13]:
features_df = pd.read_csv('features_data.csv')
features_df

,risk_tolerance,investment_experience,liquidity_needs,platform,time_spent,instrument_type_first_traded,first_deposit_amount,time_horizon,user_id
0,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,33.129417,stock,40.0,med_time_horizon,895044c23edc821881e87da749c01034
1,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,16.573517,stock,200.0,short_time_horizon,458b1d95441ced242949deefe8e4b638
2,med_risk_tolerance,limited_investment_exp,very_important_liq_need,iOS,10.008367,stock,25.0,long_time_horizon,c7936f653d293479e034865db9bb932f
3,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,1.031633,stock,100.0,short_time_horizon,b255d4bd6c9ba194d3a350b3e76c6393
4,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.187250,stock,20.0,long_time_horizon,4a168225e89375b8de605cbc0977ae91
...,...,...,...,...,...,...,...,...,...
5579,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.339283,stock,300.0,long_time_horizon,03880c726d8a4e5db006afe4119ad974
5580,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,7.241383,stock,100.0,short_time_horizon,ae8315109657f44852b24c6bca4decd6
5581,med_risk_tolerance,no_investment_exp,very_important_liq_need,both,22.967167,stock,50.0,short_time_horizon,f29c174989f9737058fe808fcf264135
5582,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,10.338417,stock,100.0,long_time_horizon,24843497d1de88b2e7233f694436cb3a


In [14]:
features_df['churned'] = features_df['user_id'].isin(churned_users_df['user_id']).astype(int)
features_df

,risk_tolerance,investment_experience,liquidity_needs,platform,time_spent,instrument_type_first_traded,first_deposit_amount,time_horizon,user_id,churned
0,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,33.129417,stock,40.0,med_time_horizon,895044c23edc821881e87da749c01034,0
1,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,16.573517,stock,200.0,short_time_horizon,458b1d95441ced242949deefe8e4b638,0
2,med_risk_tolerance,limited_investment_exp,very_important_liq_need,iOS,10.008367,stock,25.0,long_time_horizon,c7936f653d293479e034865db9bb932f,0
3,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,1.031633,stock,100.0,short_time_horizon,b255d4bd6c9ba194d3a350b3e76c6393,0
4,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.187250,stock,20.0,long_time_horizon,4a168225e89375b8de605cbc0977ae91,0
...,...,...,...,...,...,...,...,...,...,...
5579,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.339283,stock,300.0,long_time_horizon,03880c726d8a4e5db006afe4119ad974,0
5580,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,7.241383,stock,100.0,short_time_horizon,ae8315109657f44852b24c6bca4decd6,1
5581,med_risk_tolerance,no_investment_exp,very_important_liq_need,both,22.967167,stock,50.0,short_time_horizon,f29c174989f9737058fe808fcf264135,0
5582,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,10.338417,stock,100.0,long_time_horizon,24843497d1de88b2e7233f694436cb3a,0


In [15]:
features_df.dtypes

risk_tolerance                   object
investment_experience            object
liquidity_needs                  object
platform                         object
time_spent                      float64
instrument_type_first_traded     object
first_deposit_amount            float64
time_horizon                     object
user_id                          object
churned                           int64
dtype: object

In [16]:
# Cleaning up values so that dummy column names will be cleaner.

# risk tolerance
tolerance_map = {
    'high_risk_tolerance': 'high',
    'med_risk_tolerance': 'medium',
    'low_risk_tolerance': 'low'
}
features_df['risk_tolerance'] = features_df['risk_tolerance'].replace(tolerance_map)

# experience
experience_map = {
    'limited_investment_exp': 'limited',
    'no_investment_exp': 'none',
    'good_investment_exp': 'good',
    'extensive_investment_exp': 'extensive'
}
features_df['investment_experience'] = features_df['investment_experience'].replace(experience_map)

# liquidity
liquidity_map = {
    'very_important_liq_need': 'very_important',
    'somewhat_important_liq_need': 'somewhat_important',
    'not_important_liq_need': 'not_important'
}
features_df['liquidity_needs'] = features_df['liquidity_needs'].replace(liquidity_map)

# time horizon
horizon_map = {
    'short_time_horizon': 'short',
    'long_time_horizon': 'long',
    'med_time_horizon': 'medium'
}
features_df['time_horizon'] = features_df['time_horizon'].replace(horizon_map)

In [17]:
# Reduce uncommon instruments traded into one 'other' category
values_to_replace = ['cef', 'wrt', '0', 'rlt', 'lp', 'tracking']
replacement_value = 'other'
features_df['instrument_type_first_traded'] = features_df['instrument_type_first_traded'].replace(values_to_replace, replacement_value)

features_df

,risk_tolerance,investment_experience,liquidity_needs,platform,time_spent,instrument_type_first_traded,first_deposit_amount,time_horizon,user_id,churned
0,high,limited,very_important,Android,33.129417,stock,40.0,medium,895044c23edc821881e87da749c01034,0
1,medium,limited,very_important,Android,16.573517,stock,200.0,short,458b1d95441ced242949deefe8e4b638,0
2,medium,limited,very_important,iOS,10.008367,stock,25.0,long,c7936f653d293479e034865db9bb932f,0
3,medium,limited,very_important,Android,1.031633,stock,100.0,short,b255d4bd6c9ba194d3a350b3e76c6393,0
4,high,limited,very_important,Android,8.187250,stock,20.0,long,4a168225e89375b8de605cbc0977ae91,0
...,...,...,...,...,...,...,...,...,...,...
5579,high,limited,very_important,Android,8.339283,stock,300.0,long,03880c726d8a4e5db006afe4119ad974,0
5580,medium,limited,somewhat_important,iOS,7.241383,stock,100.0,short,ae8315109657f44852b24c6bca4decd6,1
5581,medium,none,very_important,both,22.967167,stock,50.0,short,f29c174989f9737058fe808fcf264135,0
5582,medium,limited,somewhat_important,iOS,10.338417,stock,100.0,long,24843497d1de88b2e7233f694436cb3a,0


In [18]:
# Get Dummies for non numeric columns, except for user_id which will be ignored in modeling
non_numeric_cols = features_df.select_dtypes(exclude=[np.number, bool]).columns.tolist()
col_to_exclude = 'user_id'
cols_for_dummies = [col for col in non_numeric_cols if col != col_to_exclude]

features_with_dummies = pd.get_dummies(features_df, columns=cols_for_dummies, dtype=int)

In [19]:
features_with_dummies.columns

Index(['time_spent', 'first_deposit_amount', 'user_id', 'churned',
       'risk_tolerance_high', 'risk_tolerance_low', 'risk_tolerance_medium',
       'investment_experience_extensive', 'investment_experience_good',
       'investment_experience_limited', 'investment_experience_none',
       'liquidity_needs_not_important', 'liquidity_needs_somewhat_important',
       'liquidity_needs_very_important', 'platform_Android', 'platform_both',
       'platform_iOS', 'instrument_type_first_traded_adr',
       'instrument_type_first_traded_etp', 'instrument_type_first_traded_mlp',
       'instrument_type_first_traded_other',
       'instrument_type_first_traded_reit',
       'instrument_type_first_traded_stock', 'time_horizon_long',
       'time_horizon_medium', 'time_horizon_short'],
      dtype='object')

In [20]:
features_with_dummies

,time_spent,first_deposit_amount,user_id,churned,risk_tolerance_high,risk_tolerance_low,risk_tolerance_medium,investment_experience_extensive,investment_experience_good,investment_experience_limited,...,platform_iOS,instrument_type_first_traded_adr,instrument_type_first_traded_etp,instrument_type_first_traded_mlp,instrument_type_first_traded_other,instrument_type_first_traded_reit,instrument_type_first_traded_stock,time_horizon_long,time_horizon_medium,time_horizon_short
0,33.129417,40.0,895044c23edc821881e87da749c01034,0,1,0,0,0,0,1,...,0,0,0,0,0,0,1,0,1,0
1,16.573517,200.0,458b1d95441ced242949deefe8e4b638,0,0,0,1,0,0,1,...,0,0,0,0,0,0,1,0,0,1
2,10.008367,25.0,c7936f653d293479e034865db9bb932f,0,0,0,1,0,0,1,...,1,0,0,0,0,0,1,1,0,0
3,1.031633,100.0,b255d4bd6c9ba194d3a350b3e76c6393,0,0,0,1,0,0,1,...,0,0,0,0,0,0,1,0,0,1
4,8.187250,20.0,4a168225e89375b8de605cbc0977ae91,0,1,0,0,0,0,1,...,0,0,0,0,0,0,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5579,8.339283,300.0,03880c726d8a4e5db006afe4119ad974,0,1,0,0,0,0,1,...,0,0,0,0,0,0,1,1,0,0
5580,7.241383,100.0,ae8315109657f44852b24c6bca4decd6,1,0,0,1,0,0,1,...,1,0,0,0,0,0,1,0,0,1
5581,22.967167,50.0,f29c174989f9737058fe808fcf264135,0,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,1
5582,10.338417,100.0,24843497d1de88b2e7233f694436cb3a,0,0,0,1,0,0,1,...,1,0,0,0,0,0,1,1,0,0


# Adding more columns I experimented with in Part 1

In [21]:
# Date of each user's first trade
features_with_dummies['first_trade'] = (
    equity_df
    .groupby('user_id')['date']
    .transform('min')
)

# Date of each user's last trade
features_with_dummies['last_trade'] = (
    equity_df
    .groupby('user_id')['date']
    .transform('max')
)

features_with_dummies['first_trade'] = pd.to_datetime(features_with_dummies['first_trade'])
features_with_dummies['last_trade'] = pd.to_datetime(features_with_dummies['last_trade'])


# Total days trading
# Redundant as no user has multiple trades in a single day
# features_with_dummies['total_active_days'] = features_with_dummies['user_id'].map(
#     equity_df.groupby('user_id')['date'].nunique()
# )

# Total trading span
features_with_dummies['trading_span'] = (features_with_dummies['last_trade'] - features_with_dummies['first_trade']).dt.days

# Total trades
features_with_dummies['total_trades'] = (
    features_with_dummies['user_id']
    .map(equity_df.groupby('user_id')['timestamp'].count())
)

# Time per trade
features_with_dummies['time_per_trade'] = features_with_dummies['time_spent'] / features_with_dummies['total_trades']

# Trades per day
features_with_dummies['trades_per_day'] = features_with_dummies['trading_span'] / features_with_dummies['total_trades']

# Trades per active day
# Redundant as no user has multiple trades in a single day
# features_with_dummies['trades_per_active_day'] = features_with_dummies['total_trades'] / features_with_dummies['total_active_days']

features_with_dummies

,time_spent,first_deposit_amount,user_id,churned,risk_tolerance_high,risk_tolerance_low,risk_tolerance_medium,investment_experience_extensive,investment_experience_good,investment_experience_limited,...,instrument_type_first_traded_stock,time_horizon_long,time_horizon_medium,time_horizon_short,first_trade,last_trade,trading_span,total_trades,time_per_trade,trades_per_day
0,33.129417,40.0,895044c23edc821881e87da749c01034,0,1,0,0,0,0,1,...,1,0,1,0,2016-08-18,2017-08-17,364,190,0.174365,1.915789
1,16.573517,200.0,458b1d95441ced242949deefe8e4b638,0,0,0,1,0,0,1,...,1,0,0,1,2016-08-18,2017-08-17,364,252,0.065768,1.444444
2,10.008367,25.0,c7936f653d293479e034865db9bb932f,0,0,0,1,0,0,1,...,1,1,0,0,2016-08-18,2017-08-17,364,252,0.039716,1.444444
3,1.031633,100.0,b255d4bd6c9ba194d3a350b3e76c6393,0,0,0,1,0,0,1,...,1,0,0,1,2016-08-18,2017-08-17,364,143,0.007214,2.545455
4,8.187250,20.0,4a168225e89375b8de605cbc0977ae91,0,1,0,0,0,0,1,...,1,1,0,0,2016-08-18,2017-08-17,364,252,0.032489,1.444444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5579,8.339283,300.0,03880c726d8a4e5db006afe4119ad974,0,1,0,0,0,0,1,...,1,1,0,0,2016-08-19,2017-08-18,364,252,0.033092,1.444444
5580,7.241383,100.0,ae8315109657f44852b24c6bca4decd6,1,0,0,1,0,0,1,...,1,0,0,1,2016-08-19,2017-08-18,364,99,0.073145,3.676768
5581,22.967167,50.0,f29c174989f9737058fe808fcf264135,0,0,0,1,0,0,0,...,1,0,0,1,2016-08-19,2017-08-18,364,197,0.116585,1.847716
5582,10.338417,100.0,24843497d1de88b2e7233f694436cb3a,0,0,0,1,0,0,1,...,1,1,0,0,2016-08-19,2017-08-18,364,141,0.073322,2.581560


In [22]:
# Aggregate daily_equity_change per user
equity_stats = full_equity_df.groupby('user_id')['daily_equity_change'].agg(
    max_daily_change='max',
    min_daily_change='min',
    max_daily_abs_change=lambda x: x.abs().max(),
    mean_daily_change='mean'
).reset_index()

# Merge into features_df
features_with_dummies = features_with_dummies.merge(equity_stats, on='user_id', how='left')


In [23]:
features_with_dummies

,time_spent,first_deposit_amount,user_id,churned,risk_tolerance_high,risk_tolerance_low,risk_tolerance_medium,investment_experience_extensive,investment_experience_good,investment_experience_limited,...,first_trade,last_trade,trading_span,total_trades,time_per_trade,trades_per_day,max_daily_change,min_daily_change,max_daily_abs_change,mean_daily_change
0,33.129417,40.0,895044c23edc821881e87da749c01034,0,1,0,0,0,0,1,...,2016-08-18,2017-08-17,364,190,0.174365,1.915789,79.22,-76.30,79.22,0.154909
1,16.573517,200.0,458b1d95441ced242949deefe8e4b638,0,0,0,1,0,0,1,...,2016-08-18,2017-08-17,364,252,0.065768,1.444444,385.92,-384.27,385.92,-0.858466
2,10.008367,25.0,c7936f653d293479e034865db9bb932f,0,0,0,1,0,0,1,...,2016-08-18,2017-08-17,364,252,0.039716,1.444444,40.24,-40.24,40.24,-0.110247
3,1.031633,100.0,b255d4bd6c9ba194d3a350b3e76c6393,0,0,0,1,0,0,1,...,2016-08-18,2017-08-17,364,143,0.007214,2.545455,203.14,-185.86,203.14,0.432598
4,8.187250,20.0,4a168225e89375b8de605cbc0977ae91,0,1,0,0,0,0,1,...,2016-08-18,2017-08-17,364,252,0.032489,1.444444,468.48,-463.73,468.48,0.285055
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5579,8.339283,300.0,03880c726d8a4e5db006afe4119ad974,0,1,0,0,0,0,1,...,2016-08-19,2017-08-18,364,252,0.033092,1.444444,3087.72,-3087.56,3087.72,-6.295890
5580,7.241383,100.0,ae8315109657f44852b24c6bca4decd6,1,0,0,1,0,0,1,...,2016-08-19,2017-08-18,364,99,0.073145,3.676768,618.27,-619.53,619.53,1.696723
5581,22.967167,50.0,f29c174989f9737058fe808fcf264135,0,0,0,1,0,0,0,...,2016-08-19,2017-08-18,364,197,0.116585,1.847716,449.21,-446.45,449.21,1.431568
5582,10.338417,100.0,24843497d1de88b2e7233f694436cb3a,0,0,0,1,0,0,1,...,2016-08-19,2017-08-18,364,141,0.073322,2.581560,365.25,-361.51,365.25,1.130850


In [24]:
# Balance on last active day
# Ensure correct ordering
full_equity_df = full_equity_df.sort_values(['user_id', 'date'])

# Keep only active days
last_active_balance = (
    full_equity_df[full_equity_df['active_day'] == 1]
    .groupby('user_id', as_index=False)
    .last()[['user_id', 'close_equity']]
    .rename(columns={'close_equity': 'balance_on_last_active_day'})
)

features_with_dummies = features_with_dummies.merge(
    last_active_balance,
    on='user_id',
    how='left'
)

In [25]:
features_with_dummies

,time_spent,first_deposit_amount,user_id,churned,risk_tolerance_high,risk_tolerance_low,risk_tolerance_medium,investment_experience_extensive,investment_experience_good,investment_experience_limited,...,last_trade,trading_span,total_trades,time_per_trade,trades_per_day,max_daily_change,min_daily_change,max_daily_abs_change,mean_daily_change,balance_on_last_active_day
0,33.129417,40.0,895044c23edc821881e87da749c01034,0,1,0,0,0,0,1,...,2017-08-17,364,190,0.174365,1.915789,79.22,-76.30,79.22,0.154909,81.22
1,16.573517,200.0,458b1d95441ced242949deefe8e4b638,0,0,0,1,0,0,1,...,2017-08-17,364,252,0.065768,1.444444,385.92,-384.27,385.92,-0.858466,382.52
2,10.008367,25.0,c7936f653d293479e034865db9bb932f,0,0,0,1,0,0,1,...,2017-08-17,364,252,0.039716,1.444444,40.24,-40.24,40.24,-0.110247,49.24
3,1.031633,100.0,b255d4bd6c9ba194d3a350b3e76c6393,0,0,0,1,0,0,1,...,2017-08-17,364,143,0.007214,2.545455,203.14,-185.86,203.14,0.432598,190.87
4,8.187250,20.0,4a168225e89375b8de605cbc0977ae91,0,1,0,0,0,0,1,...,2017-08-17,364,252,0.032489,1.444444,468.48,-463.73,468.48,0.285055,164.42
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5579,8.339283,300.0,03880c726d8a4e5db006afe4119ad974,0,1,0,0,0,0,1,...,2017-08-18,364,252,0.033092,1.444444,3087.72,-3087.56,3087.72,-6.295890,3022.58
5580,7.241383,100.0,ae8315109657f44852b24c6bca4decd6,1,0,0,1,0,0,1,...,2017-08-18,364,99,0.073145,3.676768,618.27,-619.53,619.53,1.696723,504.84
5581,22.967167,50.0,f29c174989f9737058fe808fcf264135,0,0,0,1,0,0,0,...,2017-08-18,364,197,0.116585,1.847716,449.21,-446.45,449.21,1.431568,451.61
5582,10.338417,100.0,24843497d1de88b2e7233f694436cb3a,0,0,0,1,0,0,1,...,2017-08-18,364,141,0.073322,2.581560,365.25,-361.51,365.25,1.130850,321.88


In [26]:
total_active_days = (
    full_equity_df
    .groupby('user_id')['active_day']
    .sum()
    .reset_index(name='total_active_days')
)

features_with_dummies = features_with_dummies.merge(
    total_active_days,
    on='user_id',
    how='left'
)

In [27]:
features_with_dummies

,time_spent,first_deposit_amount,user_id,churned,risk_tolerance_high,risk_tolerance_low,risk_tolerance_medium,investment_experience_extensive,investment_experience_good,investment_experience_limited,...,trading_span,total_trades,time_per_trade,trades_per_day,max_daily_change,min_daily_change,max_daily_abs_change,mean_daily_change,balance_on_last_active_day,total_active_days
0,33.129417,40.0,895044c23edc821881e87da749c01034,0,1,0,0,0,0,1,...,364,190,0.174365,1.915789,79.22,-76.30,79.22,0.154909,81.22,190
1,16.573517,200.0,458b1d95441ced242949deefe8e4b638,0,0,0,1,0,0,1,...,364,252,0.065768,1.444444,385.92,-384.27,385.92,-0.858466,382.52,252
2,10.008367,25.0,c7936f653d293479e034865db9bb932f,0,0,0,1,0,0,1,...,364,252,0.039716,1.444444,40.24,-40.24,40.24,-0.110247,49.24,252
3,1.031633,100.0,b255d4bd6c9ba194d3a350b3e76c6393,0,0,0,1,0,0,1,...,364,143,0.007214,2.545455,203.14,-185.86,203.14,0.432598,190.87,143
4,8.187250,20.0,4a168225e89375b8de605cbc0977ae91,0,1,0,0,0,0,1,...,364,252,0.032489,1.444444,468.48,-463.73,468.48,0.285055,164.42,252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5579,8.339283,300.0,03880c726d8a4e5db006afe4119ad974,0,1,0,0,0,0,1,...,364,252,0.033092,1.444444,3087.72,-3087.56,3087.72,-6.295890,3022.58,252
5580,7.241383,100.0,ae8315109657f44852b24c6bca4decd6,1,0,0,1,0,0,1,...,364,99,0.073145,3.676768,618.27,-619.53,619.53,1.696723,504.84,99
5581,22.967167,50.0,f29c174989f9737058fe808fcf264135,0,0,0,1,0,0,0,...,364,197,0.116585,1.847716,449.21,-446.45,449.21,1.431568,451.61,197
5582,10.338417,100.0,24843497d1de88b2e7233f694436cb3a,0,0,0,1,0,0,1,...,364,141,0.073322,2.581560,365.25,-361.51,365.25,1.130850,321.88,141


In [28]:
features_with_dummies.to_csv('features_data_with_dummies.csv', index=False)

# Did any users churn multiple times? is that worth noting?

# What about aggregates such as min, max, avg etc daily changes per user?

# What about final closing amount?

## 